# rect → sinc — 기하적 유도

`rect_W(t)` (W=2)의 푸리에 변환이 왜 `2sin(ωW)/ω = 2W·sinc(ωW/π)`인지 그림으로 직접 본다. 노트북은 두 부분:

1. **rect → sinc** (이 부분): 적분 $∫_{-W}^{W} e^{-j\omega t}\, dt$의 결과가 sinc인 이유를 *호 위 단위벡터의 머리잇기 합 = chord*로 푼다.
2. **sinc 평행이동 ↔ rect 평행이동**: linear phase property를 partial 적분 trail 비교로 시각 증명.

$$\text{rect}_W(t) = \begin{cases} 1, & |t| < W \\ 0, & \text{otherwise} \end{cases}, \quad
X(j\omega) = \int_{-W}^{W} e^{-j\omega t}\, dt$$

이 노트북에서 W = 2 고정.

---
*2026년 5월 · 연세대학교 전기전자공학부 22학번 김도현*


In [1]:
# Colab/Jupyter에서 ftvis 라이브러리 설치 (이미 설치돼 있으면 즉시 통과)
# 로컬에서 editable 설치로 쓰는 경우 이 셀은 그냥 건너뛰어도 무방.
%pip install -q git+https://github.com/dohyunKim0309/Fourier-Transform-Visualization.git

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ftvis import (
    signals, FourierAnalyzer,
    SignalPlot, WindingHelixPlot, WoundSignalPlot, ForwardIntegralPlot,
    SpectrumPlot,
    PartitionedAccumulationPlot,
    WoundSpectrumPlot,
    PartialIntegralComparisonPlot,
)

# 통일 — 다른 데모와 같은 dark 테마
_BG = "#0d1117"
_FG = "#e6edf3"
_GRID = "#30363d"
_AXIS = "#7d8590"
_GREEN = "#7ec699"
_BLUE = "#58a6ff"
_ORANGE = "#ffa657"
_PURPLE = "#bf91f3"
_DIM = "#3a4452"
_WHITE = "#f0f6fc"

W = 2.0
print(f'W = {W}  →  rect_W(t) = 1 for |t| < {W}')
print(f'X(jω) = ∫_{{-W}}^{{W}} e^(-jωt) dt')

W = 2.0  →  rect_W(t) = 1 for |t| < 2.0
X(jω) = ∫_{-W}^{W} e^(-jωt) dt


## [1] 적분의 3D 시각화 — 적분 구간을 따라 무엇이 더해지는가

푸리에 변환의 정의 그대로 따라가 본다. ω₀를 고정한 뒤,

$$X(j\omega_0) = \int_{-W}^{W} \text{rect}_W(t)\cdot e^{-j\omega_0 t}\, dt$$

피적분 함수는 *시간 도메인의 복소 신호*. 시간축 + 복소평면 (Re, Im) 3D에서 보면 무엇이 일어나는지 직접 보인다. ω₀ = 0.7 (잔여각 ω₀W = 1.4 rad ≈ 80°, 한 바퀴 못 채우는 작은 값)에서 시작.

In [3]:
omega0 = 0.7

sig_rect = signals.rect(width=2*W)
analyzer_3d = FourierAnalyzer(sig_rect, t_min=-W-1.5, t_max=W+1.5, n_samples=2000)
ws_3d = analyzer_3d.wound_at_omega(omega0)
last_idx = len(ws_3d.t) - 1

print(f'ω₀ = {omega0}  →  ω₀W = {omega0*W:.4f} rad ≈ {np.degrees(omega0*W):.1f}°')
print(f'∫ ≈ {ws_3d.final_integral.real:+.4f}{ws_3d.final_integral.imag:+.4f}j')

ω₀ = 0.7  →  ω₀W = 1.4000 rad ≈ 80.2°
∫ ≈ +2.8154+0.0000j


### 1-1. `rect_W(t)` 자체

|t|<W에서 1, 그 외 0. 시간 도메인 곡선.

In [4]:
p = SignalPlot(); p.update(ws_3d, last_idx); p.figure.show()

### 1-2. Winding function `e^(-jω₀t)`

각속도 -ω₀로 회전하는 단위 헬릭스. ω₀가 작으니 천천히 감긴다.

In [5]:
p = WindingHelixPlot(); p.update(ws_3d, last_idx); p.figure.show()

### 1-3. 곱한 결과 `rect_W(t)·e^(-jω₀t)` — 푸리에 적분의 피적분 함수

|t|<W에서만 살아 있는 헬릭스 조각 (envelope = rect, carrier = winding). |t|>W에서는 rect=0이라 곡선이 t축 위에 정확히 누움.

In [6]:
p = WoundSignalPlot(); p.update(ws_3d, last_idx); p.figure.show()

### 1-4. 누적 적분 trail = X(jω₀)

t를 -W에서 +W로 스윕하면서 `(rect·e^{-jω₀t})·dt`를 머리잇기로 더한 trail. 화살표 끝점이 곧 `X(jω₀)`. ω₀=0.7에서 trail이 휘어가며 끝점이 (Re축 양의 방향)으로 도달.

In [7]:
p = ForwardIntegralPlot(); p.update(ws_3d, last_idx); p.figure.show()

**[1] 정리**: 우리가 알고 싶은 건 다양한 ω에서의 누적 trail 끝점 = `X(jω)`. 이걸 그림에서 *직접* 풀려면 시간 도메인의 곡선 모양보다 **복소평면 단면**에서 보는 게 효율적. 다음 섹션부터 단면으로 옮긴다.

## [2] 복소평면 단면 — ω를 키우면 무엇이 바뀌나

`x(t)·e^(-jωt)`을 복소평면(Re, Im)에 그리면 호 위 점들이 된다. rect의 경우 |t|<W에서만 살아있으니, 살아있는 t 구간 [-W, +W]에서 단위벡터 `e^(-jωt_k)`들을 균등 시간 분할로 점 찍는다. 그 점들의 *벡터 합*이 X(jω) — 머리잇기로 누적한 trail 끝점.

ω가 커지면:
- 첫 점($e^{j\omega W}$)부터 끝 점($e^{-j\omega W}$)까지 **각도 차이 = 2ωW**가 커짐
- 2ωW가 2π를 넘기면 호가 한 바퀴를 넘어 다시 *되돌아오는* 구간이 생김. 그 되돌아오는 부분은 **이전 구간과 cancel**돼 결과에 기여 안 함. 잔여(못 채운 마지막) 호의 *양 끝 변위*만 X(jω)에 남음
- 균등 시간 Δt가 ω·Δt 만큼 회전하니 점들 사이 **간격이 듬성**해짐 (밀도 감소)

ω₀W = π/12, π/2, 13π/12, 25π/12 — 한 바퀴 0개, 0개, 1개, 2개를 cancel하는 케이스 비교.

In [8]:
ns_phase2 = [0, 0, 1, 2]   # 추가로 ω₀W 값 직접 명시
omega_W_vals = [(0, np.pi/12), (0, np.pi/2), (1, 13*np.pi/12), (2, 25*np.pi/12)]
ARROWS_PER_PERIOD = 24
X_MIN_P2, X_MAX_P2, Y_LIM_P2 = -1.3, 4.5, 1.3

for n, theta in omega_W_vals:
    omega = theta / W
    # 한 주기당 ARROWS_PER_PERIOD개; n주기 + 잔여
    n_total = max(ARROWS_PER_PERIOD * n + ARROWS_PER_PERIOD // 12, 12)
    dt = 2 * W / n_total
    t_k = -W + np.arange(n_total) * dt + dt / 2
    unit_vecs = np.exp(-1j * omega * t_k)

    # 한 주기 cancel되는 부분의 단위벡터 수
    if omega > 1e-9:
        one_period_arrows = 2 * np.pi / (omega * dt)
        boundary = int(round(n * one_period_arrows))
    else:
        boundary = 0

    label = (f'ω₀W = {theta/np.pi:.3f}π ({np.degrees(theta):.1f}°),  '
             f'n={n} 바퀴 cancel + 잔여')
    panel = PartitionedAccumulationPlot()
    panel.show_partition(unit_vecs, dt, boundary=boundary, label=label,
                         x_range=(X_MIN_P2, X_MAX_P2),
                         y_range=(-Y_LIM_P2, Y_LIM_P2))
    panel.figure.show()

**[2] 정리**:
- ω가 작은 첫 번째 케이스: 잔여 호가 짧음 → 끝점이 (3.99, 0) 근처 (≈ 2W)
- ω가 커진 둘째 케이스: 잔여각 π/2까지 — 끝점이 chord/ω 비례로 줄어듦
- 셋·넷째 케이스: 한 바퀴를 다 도는 부분(회색 dim)이 cancel되어 끝점에 기여 안 함. **결과를 결정하는 건 잔여 호의 양 끝점 변위뿐**

이 마지막 메시지가 결정적: *호 모양이 어떻게 생겼든, 결과에 남는 건 양 끝점 변위*. 다음 섹션에서 이걸 정확히 풀 것.

## [3] 미소 벡터의 정체 — 호의 접선

[2]에서 "결과 = 잔여 호의 양 끝점 변위"라는 직관을 봤다. 이걸 식으로 풀려면, 우선 변수 변환.

$$\int_{-W}^{W} e^{-j\omega t}\, dt \;\stackrel{\theta = -\omega t}{=}\; \frac{1}{\omega}\int_{-\omega W}^{+\omega W} e^{j\theta}\, d\theta$$

`1/ω`는 *시간↔각도 변환의 야코비안*. 일단 빼두고 **`∫e^(jθ)dθ` 자체에 집중**한다.

핵심: **`e^(jθ)·dθ`는 단위원 위 점의 미소 *접선* 변위와 무엇이 다른가?**

단위원 위 점 `e^(jθ)`를 θ에 대해 미분하면

$$\frac{d}{d\theta}\bigl(e^{j\theta}\bigr) = j\,e^{j\theta} \;\;\Longleftrightarrow\;\; d(e^{j\theta}) = j\,e^{j\theta}\, d\theta$$

즉 **단위원 위 점의 미소 변위 $d(e^{j\theta})$는 그 점의 위치 벡터 $e^{j\theta}$를 90° 반시계 회전시킨 방향**(j 곱). 이게 바로 호의 접선.

뒤집으면

$$e^{j\theta}\, d\theta = \frac{1}{j}\, d(e^{j\theta})$$

우리가 합하려는 항 `e^(jθ)·dθ`는 **호 위 미소 변위 / j**. 즉 미소 변위를 90° *시계* 회전시킨 것.

다음 그림에서 한 점 $P = e^{j\theta_0}$에서 두 화살표를 같이 본다.

In [9]:
theta0 = np.pi / 4   # 호 위 한 점
dtheta_demo = np.pi / 8  # 시각적으로 알아볼 만한 큰 dθ
P = np.exp(1j * theta0)
Q = np.exp(1j * (theta0 + dtheta_demo))

fig = go.Figure()

# 단위원
th_full = np.linspace(0, 2*np.pi, 200)
fig.add_trace(go.Scatter(x=np.cos(th_full), y=np.sin(th_full), mode='lines',
                         line=dict(color=_AXIS, width=1, dash='dot'),
                         hoverinfo='skip', showlegend=False))

# 호 PQ
th_arc = np.linspace(theta0, theta0 + dtheta_demo, 30)
fig.add_trace(go.Scatter(x=np.cos(th_arc), y=np.sin(th_arc), mode='lines',
                         line=dict(color=_BLUE, width=4),
                         hoverinfo='skip', showlegend=False))

# P점 (위치 벡터 끝)
fig.add_trace(go.Scatter(x=[P.real], y=[P.imag], mode='markers+text',
                         marker=dict(color=_GREEN, size=10),
                         text=['P'], textposition='top right',
                         textfont=dict(color=_GREEN, size=14),
                         hoverinfo='skip', showlegend=False))

# Q점
fig.add_trace(go.Scatter(x=[Q.real], y=[Q.imag], mode='markers+text',
                         marker=dict(color=_BLUE, size=10),
                         text=['Q'], textposition='top right',
                         textfont=dict(color=_BLUE, size=14),
                         hoverinfo='skip', showlegend=False))

# 위치 벡터 OP = e^(jθ₀)
fig.add_trace(go.Scatter(x=[0, P.real], y=[0, P.imag], mode='lines',
                         line=dict(color=_GREEN, width=2, dash='dot'),
                         hoverinfo='skip', showlegend=False))

# 우리가 합하려는 화살표: P 위치에 e^(jθ₀)·dθ 크기로
# (= P 자체 방향, 길이 dθ)
arrow_target = P * dtheta_demo
fig.add_trace(go.Scatter(x=[P.real, P.real + arrow_target.real],
                         y=[P.imag, P.imag + arrow_target.imag],
                         mode='lines+markers',
                         line=dict(color=_GREEN, width=4),
                         marker=dict(color=_GREEN, size=[3, 9], symbol='arrow-up'),
                         hoverinfo='skip', showlegend=False))
# 라벨
fig.add_annotation(x=P.real + arrow_target.real * 1.2, y=P.imag + arrow_target.imag * 1.2,
                   text='e^(jθ₀)·dθ<br>(우리가 합할 항)',
                   font=dict(color=_GREEN, size=11),
                   showarrow=False)

# 미소 변위 d(e^(jθ)) = j·e^(jθ)·dθ — P에서 호의 접선 방향
arrow_disp = 1j * P * dtheta_demo
fig.add_trace(go.Scatter(x=[P.real, P.real + arrow_disp.real],
                         y=[P.imag, P.imag + arrow_disp.imag],
                         mode='lines+markers',
                         line=dict(color=_ORANGE, width=4),
                         marker=dict(color=_ORANGE, size=[3, 9], symbol='arrow-up'),
                         hoverinfo='skip', showlegend=False))
fig.add_annotation(x=P.real + arrow_disp.real * 1.4, y=P.imag + arrow_disp.imag * 1.4,
                   text='d(e^(jθ)) = j·e^(jθ₀)·dθ<br>(호의 접선, P→Q)',
                   font=dict(color=_ORANGE, size=11),
                   showarrow=False)

# 직각 표시 — 두 화살표 사이
def right_angle_marker(origin, dir1, dir2, size=0.06):
    # origin에서 dir1·size, dir1·size + dir2·size, dir2·size 세 점으로 ㄱ자
    u1 = dir1 / abs(dir1) * size
    u2 = dir2 / abs(dir2) * size
    return [origin + u1, origin + u1 + u2, origin + u2]

ra_pts = right_angle_marker(P, arrow_target, arrow_disp)
fig.add_trace(go.Scatter(x=[p.real for p in ra_pts], y=[p.imag for p in ra_pts],
                         mode='lines',
                         line=dict(color=_FG, width=1.5),
                         hoverinfo='skip', showlegend=False))
# 직각 라벨
fig.add_annotation(x=P.real + 0.12*np.cos(theta0+np.pi/4),
                   y=P.imag + 0.12*np.sin(theta0+np.pi/4),
                   text='90°',
                   font=dict(color=_FG, size=10), showarrow=False)

# dθ 각도 호 (원점 중심, 작은 반지름)
th_dt_arc = np.linspace(theta0, theta0 + dtheta_demo, 20)
r_arc = 0.18
fig.add_trace(go.Scatter(x=r_arc*np.cos(th_dt_arc), y=r_arc*np.sin(th_dt_arc),
                         mode='lines',
                         line=dict(color=_PURPLE, width=2),
                         hoverinfo='skip', showlegend=False))
fig.add_annotation(x=r_arc*1.7*np.cos(theta0 + dtheta_demo/2),
                   y=r_arc*1.7*np.sin(theta0 + dtheta_demo/2),
                   text='dθ',
                   font=dict(color=_PURPLE, size=12), showarrow=False)

fig.update_layout(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    font=dict(color=_FG, size=11),
    margin=dict(l=10, r=10, t=50, b=10),
    title=dict(text='[3-1] 호 위 한 점에서: 위치 벡터·dθ vs 호 접선 미소 변위',
               x=0.02, xanchor='left'),
    width=600, height=560, showlegend=False,
    xaxis=dict(range=[-0.3, 1.7], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, scaleanchor='y', scaleratio=1, title='Re'),
    yaxis=dict(range=[-0.3, 1.7], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, title='Im'),
)
fig.show()

**[3-1] 정리**: 점 P에서 두 화살표가 정확히 90° 차이.

- **초록 ($e^{j\theta_0}\cdot d\theta$)**: 합할 항. 위치 벡터 P 방향, 길이 dθ.
- **주황 ($j\cdot e^{j\theta_0}\cdot d\theta = d(e^{j\theta})$)**: 호의 접선, P→Q 방향. 미소 변위.

핵심: **미소 변위들을 머리잇기로 더하면 호 자체를 그린다**. 시작점에서 끝점까지의 *변위 = chord*가 미소 변위 합. 그러면 우리가 진짜 합하려는 항(= 미소 변위 / j = 90° 시계 회전한 것)의 합도 같은 chord를 90° 회전한 것.

이제 호 위 N개 점에서 같이 보자.

In [10]:
N_ARROWS = 12
theta_W = np.pi / 6   # ω₀W = π/6, 호 양 끝각 ±π/6
theta_grid = np.linspace(-theta_W, theta_W, N_ARROWS)
unit_pts = np.exp(1j * theta_grid)
dtheta_arrows = 2 * theta_W / N_ARROWS

fig = go.Figure()

# 단위원
fig.add_trace(go.Scatter(x=np.cos(th_full), y=np.sin(th_full), mode='lines',
                         line=dict(color=_AXIS, width=1, dash='dot'),
                         hoverinfo='skip', showlegend=False))

# 호 — 전체 호 위 굵은 곡선
th_arc = np.linspace(-theta_W, theta_W, 80)
fig.add_trace(go.Scatter(x=np.cos(th_arc), y=np.sin(th_arc), mode='lines',
                         line=dict(color=_BLUE, width=3),
                         hoverinfo='skip', showlegend=False))

# 호 위 점들
fig.add_trace(go.Scatter(x=unit_pts.real, y=unit_pts.imag, mode='markers',
                         marker=dict(color=_GREEN, size=6),
                         hoverinfo='skip', showlegend=False))

# 각 점에서 e^(jθ_k)·dθ 화살표 (위치 벡터 방향, 작게)
for pt in unit_pts:
    arrow = pt * dtheta_arrows
    fig.add_trace(go.Scatter(x=[pt.real, pt.real + arrow.real],
                             y=[pt.imag, pt.imag + arrow.imag],
                             mode='lines',
                             line=dict(color=_GREEN, width=1.5),
                             opacity=0.9, hoverinfo='skip', showlegend=False))

# 머리잇기 누적 합 — 원점에서 시작
arrows_seq = unit_pts * dtheta_arrows
cum = np.concatenate([[0+0j], np.cumsum(arrows_seq)])
fig.add_trace(go.Scatter(x=cum.real, y=cum.imag, mode='lines',
                         line=dict(color=_GREEN, width=4),
                         hoverinfo='skip', showlegend=False))

# 합 = 누적 끝점 (chord 방향, 원점에서 끝점까지)
sum_vec = cum[-1]
fig.add_trace(go.Scatter(x=[0, sum_vec.real], y=[0, sum_vec.imag],
                         mode='lines+markers',
                         line=dict(color=_ORANGE, width=4),
                         marker=dict(color=_ORANGE, size=[3, 11], symbol='arrow-up'),
                         hoverinfo='skip', showlegend=False))
fig.add_annotation(x=sum_vec.real * 0.55, y=sum_vec.imag - 0.1,
                   text=f'∫ e^(jθ) dθ ≈ {sum_vec.real:.3f}{sum_vec.imag:+.3f}j',
                   font=dict(color=_ORANGE, size=11), showarrow=False)

# 호 양 끝점 강조 + chord (위치 벡터 끝 두 점 잇기)
p_start = np.exp(1j * (-theta_W))
p_end = np.exp(1j * theta_W)
fig.add_trace(go.Scatter(x=[p_start.real, p_end.real],
                         y=[p_start.imag, p_end.imag],
                         mode='lines+markers',
                         line=dict(color=_PURPLE, width=2, dash='dash'),
                         marker=dict(color=_PURPLE, size=8),
                         hoverinfo='skip', showlegend=False))
fig.add_annotation(x=(p_start.real + p_end.real)/2 + 0.1,
                   y=(p_start.imag + p_end.imag)/2,
                   text='호 양 끝 변위<br>(j·∫e^(jθ)dθ)',
                   font=dict(color=_PURPLE, size=11), showarrow=False)

fig.update_layout(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    font=dict(color=_FG, size=11),
    margin=dict(l=10, r=10, t=50, b=10),
    title=dict(text=f'[3-2] 호 위 N={N_ARROWS}개 단위벡터의 합 (ω₀W = π/6)',
               x=0.02, xanchor='left'),
    width=620, height=560, showlegend=False,
    xaxis=dict(range=[-0.3, 1.4], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, scaleanchor='y', scaleratio=1, title='Re'),
    yaxis=dict(range=[-0.85, 0.85], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, title='Im'),
)
fig.show()

print(f'합 ∫e^(jθ)dθ ≈ {sum_vec.real:.4f}{sum_vec.imag:+.4f}j')
print(f'이론값 2sin(ωW) = 2sin(π/6) = {2*np.sin(np.pi/6):.4f}  (실수, Re축 위)')
print(f'호 양 끝 변위 (chord) = e^(j·π/6) - e^(-j·π/6) = {(p_end - p_start).real:+.4f}{(p_end - p_start).imag:+.4f}j')
print(f'  → 순허수 2j·sin(π/6) = {2j*np.sin(np.pi/6):.4f}j')

합 ∫e^(jθ)dθ ≈ 0.9915+0.0000j
이론값 2sin(ωW) = 2sin(π/6) = 1.0000  (실수, Re축 위)
호 양 끝 변위 (chord) = e^(j·π/6) - e^(-j·π/6) = +0.0000+1.0000j
  → 순허수 2j·sin(π/6) = 0.0000+1.0000jj


**[3-2] 정리**: 호 위 N개 단위벡터(초록 화살표) 합이 *호와 평행*하게 자라며 끝점에 도달.

- **합 = 2sin(ωW)** — 실수, Re축 위
- **호 양 끝 변위** (보라 점선) = $e^{j\omega W} - e^{-j\omega W} = 2j\sin(\omega W)$ — *순허수*
- 두 결과가 정확히 90° 차이 — 우리 식 $e^{j\theta}d\theta = (1/j)\cdot d(e^{j\theta})$와 일치 (1/j = -j = 90° 시계 회전)

미소 벡터 합과 chord (호 양 끝 변위)은 90° 차이로 같은 정보를 담음. 호의 *모양*은 무관, 양 끝점만 결과에 기여.

## [4] sin(ωW) — chord 길이의 기하적 유도

[3-2]에서 호 양 끝 변위가 $e^{j\omega W} - e^{-j\omega W}$임을 봤다. 그 *길이*가 정확히 $2\sin(\omega W)$라는 게 단위원 + 직각 삼각형으로 직접 보인다.

**기하 유도**:
- 단위원 위 두 점 $A = e^{j\omega W}$ (각도 +ωW), $B = e^{-j\omega W}$ (각도 -ωW). 둘은 Re축에 대해 켤레.
- 두 점은 Re축에서 같은 거리에 위치: $cos(\omega W)$. 즉 두 점의 Re 좌표는 같다.
- A의 Im 좌표: $+\sin(\omega W)$,  B의 Im 좌표: $-\sin(\omega W)$.
- 두 점을 잇는 직선은 **Re축에 수직**. 길이 = $\sin(\omega W) - (-\sin(\omega W)) = 2\sin(\omega W)$.

따라서

$$\big|e^{j\omega W} - e^{-j\omega W}\big| = 2\sin(\omega W)$$

그리고 이 변위는 *순허수* 방향 (Re축에 수직), 부호는 ωW의 부호에 따라 ±2j·sin(ωW). 1/j을 곱하면 (= 90° 시계 회전) 실수 `2sin(ωW)`. 즉

$$\int_{-\omega W}^{+\omega W} e^{j\theta}\, d\theta = \frac{1}{j}\cdot 2j\sin(\omega W) = 2\sin(\omega W)$$

다음 그림은 이 유도를 시각적으로 확인.

In [11]:
theta_W_demo = np.pi / 3
A = np.exp(1j * theta_W_demo)
B = np.exp(-1j * theta_W_demo)
M = (A + B) / 2  # Re축 위 중점

fig = go.Figure()

# 단위원
fig.add_trace(go.Scatter(x=np.cos(th_full), y=np.sin(th_full), mode='lines',
                         line=dict(color=_AXIS, width=1, dash='dot'),
                         hoverinfo='skip', showlegend=False))

# 위치 벡터 OA, OB
for label, pt, color in [('A=e^(jωW)', A, _GREEN), ('B=e^(-jωW)', B, _BLUE)]:
    fig.add_trace(go.Scatter(x=[0, pt.real], y=[0, pt.imag], mode='lines',
                             line=dict(color=color, width=2),
                             hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=[pt.real], y=[pt.imag], mode='markers+text',
                             marker=dict(color=color, size=10),
                             text=[label], textposition='middle right',
                             textfont=dict(color=color, size=12),
                             hoverinfo='skip', showlegend=False))

# A에서 Re축으로 내린 수선
fig.add_trace(go.Scatter(x=[A.real, A.real], y=[A.imag, 0], mode='lines',
                         line=dict(color=_GRID, width=1.5, dash='dot'),
                         hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(x=[B.real, B.real], y=[B.imag, 0], mode='lines',
                         line=dict(color=_GRID, width=1.5, dash='dot'),
                         hoverinfo='skip', showlegend=False))

# 직각 표시 (A의 Re축 발 지점, M 위치)
ra_size = 0.05
fig.add_trace(go.Scatter(x=[M.real - ra_size, M.real - ra_size, M.real],
                         y=[ra_size, 0, 0],
                         mode='lines',
                         line=dict(color=_FG, width=1.5),
                         hoverinfo='skip', showlegend=False))

# AB chord (호 양 끝 변위) — A에서 B로
fig.add_trace(go.Scatter(x=[A.real, B.real], y=[A.imag, B.imag], mode='lines+markers',
                         line=dict(color=_PURPLE, width=4),
                         marker=dict(color=_PURPLE, size=[7, 7]),
                         hoverinfo='skip', showlegend=False))

# AM 라벨 = sin(ωW)
fig.add_annotation(x=A.real + 0.06, y=A.imag/2,
                   text='sin(ωW)',
                   font=dict(color=_PURPLE, size=12), showarrow=False)
fig.add_annotation(x=B.real + 0.06, y=B.imag/2,
                   text='sin(ωW)',
                   font=dict(color=_PURPLE, size=12), showarrow=False)
# AB 길이 라벨
fig.add_annotation(x=A.real + 0.18, y=0,
                   text=f'|AB| = 2sin(ωW)<br>= 2sin({theta_W_demo/np.pi:.3f}π)<br>≈ {2*np.sin(theta_W_demo):.3f}',
                   font=dict(color=_PURPLE, size=12), showarrow=False)

# ωW 각도 호 (원점, A까지)
th_arc_label = np.linspace(0, theta_W_demo, 30)
fig.add_trace(go.Scatter(x=0.25*np.cos(th_arc_label), y=0.25*np.sin(th_arc_label),
                         mode='lines',
                         line=dict(color=_GREEN, width=2),
                         hoverinfo='skip', showlegend=False))
fig.add_annotation(x=0.32*np.cos(theta_W_demo/2), y=0.32*np.sin(theta_W_demo/2),
                   text='ωW',
                   font=dict(color=_GREEN, size=12), showarrow=False)
# -ωW
fig.add_trace(go.Scatter(x=0.25*np.cos(-th_arc_label), y=0.25*np.sin(-th_arc_label),
                         mode='lines',
                         line=dict(color=_BLUE, width=2),
                         hoverinfo='skip', showlegend=False))

fig.update_layout(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    font=dict(color=_FG, size=11),
    margin=dict(l=10, r=10, t=50, b=10),
    title=dict(text=f'[4] chord 길이 = 2sin(ωW) — 단위원 + 직각 삼각형',
               x=0.02, xanchor='left'),
    width=600, height=560, showlegend=False,
    xaxis=dict(range=[-0.3, 1.5], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, scaleanchor='y', scaleratio=1, title='Re'),
    yaxis=dict(range=[-1.3, 1.3], gridcolor=_GRID, zerolinecolor=_AXIS,
               color=_FG, title='Im'),
)
fig.show()

**[4] 정리**: chord 길이 = `2sin(ωW)`. 두 켤레점 A, B의 Im 좌표 차이 = 두 sin 값 사이 거리. 그 길이가 호 양 끝 변위의 크기.

부호와 방향까지 정확히:

$$e^{j\omega W} - e^{-j\omega W} = 2j\sin(\omega W)$$

순허수, 크기 $|2sin(\omega W)|$. 1/j 곱하면 실수 $2sin(\omega W)$ (음수도 가능 — ωW가 π를 넘으면 sin이 음).

## [5] ωW를 0 → 4π로 — sin이 진동하는 모습

ωW를 슬라이더로 조절하면서 호 위 단위벡터들과 그 합이 어떻게 변하는지 본다.

- ωW가 0 → π/2: 합이 0 → 2까지 자람
- ωW = π: 호가 반바퀴, 끝점 = 시작점의 정반대 (= 두 점이 같은 Re 좌표 -1) → chord = 0
- ωW = 3π/2: 호가 3/4바퀴 회전, 합이 음수 (= -2)
- ωW = 2π: 호가 한 바퀴 닫힘 → chord = 0
- 그 후 진동 반복 (sin 자체)

In [12]:
N_FRAMES = 41
theta_W_slider = np.linspace(0, 4*np.pi, N_FRAMES)
N_ARROW_SLIDER = 30

# 모든 frame을 미리 계산
frames = []
for i, twW in enumerate(theta_W_slider):
    if twW < 1e-9:
        # ωW=0 케이스: 모든 점이 1+0j, 합도 0
        unit_x = [1.0]; unit_y = [0.0]
        cum_x = [0.0, 0.0]; cum_y = [0.0, 0.0]
        sum_x = [0.0, 0.0]; sum_y = [0.0, 0.0]
        chord_x = [1.0, 1.0]; chord_y = [0.0, 0.0]
        sum_re = 0.0
    else:
        theta_grid = np.linspace(-twW, twW, N_ARROW_SLIDER)
        unit_pts = np.exp(1j * theta_grid)
        unit_x = unit_pts.real.tolist()
        unit_y = unit_pts.imag.tolist()
        dtheta = 2 * twW / N_ARROW_SLIDER
        cum = np.concatenate([[0+0j], np.cumsum(unit_pts * dtheta)])
        cum_x = cum.real.tolist()
        cum_y = cum.imag.tolist()
        sum_x = [0.0, float(cum.real[-1])]
        sum_y = [0.0, float(cum.imag[-1])]
        # chord (단위원 위 두 끝점)
        p_st = np.exp(-1j * twW)
        p_en = np.exp(1j * twW)
        chord_x = [p_st.real, p_en.real]
        chord_y = [p_st.imag, p_en.imag]
        sum_re = float(cum.real[-1])

    frames.append(go.Frame(
        data=[
            go.Scatter(x=unit_x, y=unit_y, mode='markers',
                       marker=dict(color=_GREEN, size=4)),
            go.Scatter(x=cum_x, y=cum_y, mode='lines',
                       line=dict(color=_GREEN, width=3)),
            go.Scatter(x=sum_x, y=sum_y, mode='lines+markers',
                       line=dict(color=_ORANGE, width=4),
                       marker=dict(color=_ORANGE, size=[3, 11])),
            go.Scatter(x=chord_x, y=chord_y, mode='lines',
                       line=dict(color=_PURPLE, width=2, dash='dash')),
        ],
        name=f'{twW/np.pi:.2f}π',
        layout=dict(title=dict(
            text=f'[5] ωW = {twW/np.pi:.2f}π,  ∫e^(jθ)dθ ≈ {sum_re:+.3f}'))
    ))

# 초기 프레임 — i=0
fig = go.Figure(
    data=frames[0].data,
    layout=go.Layout(
        paper_bgcolor=_BG, plot_bgcolor=_BG,
        font=dict(color=_FG, size=11),
        margin=dict(l=10, r=10, t=50, b=80),
        title=dict(text=f'[5] ωW = 0,  ∫e^(jθ)dθ ≈ 0',
                   x=0.02, xanchor='left'),
        width=600, height=620, showlegend=False,
        xaxis=dict(range=[-2.3, 2.3], gridcolor=_GRID, zerolinecolor=_AXIS,
                   color=_FG, scaleanchor='y', scaleratio=1, title='Re'),
        yaxis=dict(range=[-2.3, 2.3], gridcolor=_GRID, zerolinecolor=_AXIS,
                   color=_FG, title='Im'),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix='ωW = ', visible=True, xanchor='right'),
            pad=dict(t=40),
            steps=[dict(method='animate', label=f'{twW/np.pi:.2f}π',
                        args=[[f'{twW/np.pi:.2f}π'],
                              dict(frame=dict(duration=0, redraw=True),
                                   mode='immediate', transition=dict(duration=0))])
                   for twW in theta_W_slider],
        )],
    ),
    frames=frames,
)

# 단위원 점선을 모든 프레임 배경으로
th_full_local = np.linspace(0, 2*np.pi, 200)
fig.add_trace(go.Scatter(x=np.cos(th_full_local), y=np.sin(th_full_local),
                         mode='lines',
                         line=dict(color=_AXIS, width=1, dash='dot'),
                         hoverinfo='skip', showlegend=False))
fig.show()

**[5] 정리**: 슬라이더를 움직이며 직접 확인.

- ωW = 0, π, 2π, 3π, ... 에서 합 = 0 (호가 정수 바퀴)
- ωW = π/2, 3π/2, 5π/2 ... 에서 합 = ±2 (sin 극값)
- 한 바퀴를 다 돈 부분(=cancel)은 trail에서 작은 원으로 보임 — [2]에서 본 dim 묶음의 의미

수치 한 줄: `합 = 2sin(ωW)` 그 자체.

## [6] 1/ω 다시 곱하면 sinc

[3]~[5]에서 본 건 $\int_{-\omega W}^{+\omega W} e^{j\theta}\, d\theta = 2\sin(\omega W)$. 우리가 진짜 알고 싶었던 건

$$\int_{-W}^{W} e^{-j\omega t}\, dt = \frac{1}{\omega}\int_{-\omega W}^{+\omega W} e^{j\theta}\, d\theta = \frac{2\sin(\omega W)}{\omega}$$

`1/ω`는 시간 ↔ 각도 변환의 야코비안. 시간 dt 한 단위가 각도 ω·dt 만큼의 회전을 만드니, 같은 ωW만큼 회전하는 데 걸리는 *시간*은 W (고정). 그 시간을 ω-도메인의 적분 결과로 환산하면 `1/ω`로 스케일링.

기하적으로: ω가 작을 때는 호가 짧고 chord도 짧지만 *시간 단위*로는 W까지 길게 적분. ω가 클 때는 호가 길어지지만 같은 W 시간 안에 다 회전을 끝내니 시간당 기여가 1/ω로 줄어듦.

따라서

$$\boxed{X(j\omega) = \int_{-W}^{W} e^{-j\omega t}\, dt = \frac{2\sin(\omega W)}{\omega} = 2W\cdot\text{sinc}\!\left(\frac{\omega W}{\pi}\right)}$$

이 식의 **세 인자**는 [3]~[5]에서 본 기하 요소들의 직접 대응:

| 인자 | 출처 |
|---|---|
| `2sin(ωW)` | 호 양 끝점 변위 chord의 1/j 회전 ([3], [4]) |
| `1/ω` | 시간↔각도 변환 야코비안 |
| 부호 (sin) | 호가 한 바퀴 넘으면 cancel, 다음 반바퀴 음수 ([5]) |

## [7] 검증 — SpectrumPlot으로 X(jω) 전체 확인

ω 전 범위에서 X(jω)를 계산해 sinc 모양이 나오는지 확인.

In [13]:
sig = signals.rect(width=2*W)
analyzer = FourierAnalyzer(sig, t_min=-W-1, t_max=W+1, n_samples=4000)

omegas, X = analyzer.spectrum(omega_min=-6*np.pi, omega_max=6*np.pi, n_omega=601)

panel = SpectrumPlot(view='re_im', show_inset=True)
panel.show_spectrum(omegas, X)
panel.figure.show()

# 분석값 비교
X_analytic = 2 * np.sin(omegas * W) / np.where(np.abs(omegas) < 1e-9, 1.0, omegas)
X_analytic = np.where(np.abs(omegas) < 1e-9, 2*W, X_analytic)
err = float(np.max(np.abs(X - X_analytic)))
print(f'max|X_numerical - 2sin(ωW)/ω| = {err:.2e}')
print(f'X(0) = {X[np.argmin(np.abs(omegas))].real:.4f}  (expect 2W = {2*W})')
print(f'first zero at ω = π/W = {np.pi/W:.4f}')

max|X_numerical - 2sin(ωW)/ω| = 3.39e-06
X(0) = 4.0000  (expect 2W = 4.0)
first zero at ω = π/W = 1.5708


**관찰**:
- X(jω)가 순수 실수 — rect가 짝함수 (Im 축이 평면으로 누움; flatten_noise 작동)
- X(0) = 2W = 4 (DC 값 = 신호의 시간 적분)
- 첫 영점 ω = π/W = π/2 — [5]에서 ωW = π일 때 합이 0이 되는 것과 일치
- 이후 영점들도 ω = nπ/W에서 — 호가 정수 바퀴 닫히는 지점

**(1) rect → sinc 부분 끝.** 다음 부분에서 sinc의 평행이동(linear phase)이 시간 도메인 평행이동에 해당함을 본다.

---

# (2) sinc 회전이동 ↔ rect 평행이동

여기까지 `rect_W(t)`가 어떻게 `2sin(ωW)/ω = sinc`로 변환되는지 기하적으로 봤다. 이제 그 *역방향* — sinc의 spectrum을 다시 적분해 rect로 되돌리는 과정과, 거기에 linear phase $e^{-j\omega\alpha}$가 곱해지면 평행이동된 rect가 나온다는 time shift property를 시각 증명한다.

## (2-1) sinc → rect 역변환

지금까지 *순방향* 변환 X(jω) = ∫_{-W}^{W} e^{-jωt} dt를 기하적으로 분해했다. 이제 *역방향* — sinc를 다시 rect로 되돌려본다.

역변환:
$$
x(t_\text{fix}) = \frac{1}{2\pi}\int X(j\omega)\, e^{j\omega t_\text{fix}}\, d\omega
$$

이걸 ω별 화살표 `(1/2π)·X(jω)·e^{jωt_fix}·dω`의 머리잇기 누적으로 본다 ([02_inverse.ipynb](./02_inverse.ipynb) 참조). 누적 끝점이 곧 `x(t_fix)` = rect_W(t_fix).

**기대 결과**:
- 펄스 외부 (`|t_fix| > W`): 끝점이 0 부근.
- 펄스 경계 (`|t_fix| = W`): 끝점이 0.5 (사다리꼴 적분의 자연스러운 결과).
- 펄스 내부 (`|t_fix| < W`): 끝점이 1 부근.

펄스 내부/경계/외부 4개 t_fix에서 누적 trail의 *모양*이 어떻게 다른지 small multiples로 비교한다.

In [14]:
# Phase 3에서 만든 spectrum 데이터를 더 넓은 ω 범위로 다시 (역변환 정확도 확보)
omegas_inv, X_inv = analyzer.spectrum(omega_min=-30, omega_max=30, n_omega=601)
domega_inv = float(omegas_inv[1] - omegas_inv[0])

t_fix_panels = [-2.5, -2.0, 0.0, 1.5]   # 외부 / 경계 / 내부 / 내부
labels = ['외부 t_fix=-2.5', '경계 t_fix=-2.0', '내부 t_fix=0.0', '내부 t_fix=1.5']

# paired_by_abs 순서로 정렬 (켤레 cancel이 깨끗)
order_idx = np.argsort(np.abs(omegas_inv))
om_o = omegas_inv[order_idx]
X_o = X_inv[order_idx]

fig = go.Figure()
fig.set_subplots(rows=1, cols=4, horizontal_spacing=0.05,
                 subplot_titles=labels)

# 모든 패널 동일한 축 범위 — 각 t_fix에서 누적 끝점/trail의 *상대적 크기*가 직접 비교됨
all_re_ranges = []
all_im_ranges = []
accums = []
for t_fix in t_fix_panels:
    arrows = X_o * np.exp(1j * om_o * t_fix) * domega_inv / (2 * np.pi)
    accum = np.cumsum(arrows)
    accums.append((arrows, accum, t_fix))
    all_re_ranges.append((accum.real.min(), accum.real.max()))
    all_im_ranges.append((accum.imag.min(), accum.imag.max()))

# 모든 trail이 들어오는 정사각 범위
re_max = max(max(abs(lo), abs(hi)) for lo, hi in all_re_ranges)
im_max = max(max(abs(lo), abs(hi)) for lo, hi in all_im_ranges)
lim_p4 = max(re_max, im_max, 1.0) * 1.2

for col, (arrows, accum, t_fix) in enumerate(accums, start=1):
    # 화살표 줄기 (None separator)
    starts = accum - arrows
    ends = accum
    n = len(arrows)
    xs = np.empty(3*n); ys = np.empty(3*n)
    xs[0::3] = starts.real; xs[1::3] = ends.real; xs[2::3] = np.nan
    ys[0::3] = starts.imag; ys[1::3] = ends.imag; ys[2::3] = np.nan
    fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines',
                             line=dict(color=_ORANGE, width=1.2),
                             opacity=0.45, hoverinfo='skip', showlegend=False),
                  row=1, col=col)
    # 누적 trail
    fig.add_trace(go.Scatter(x=accum.real, y=accum.imag, mode='lines',
                             line=dict(color=_GREEN, width=2.5),
                             hoverinfo='skip', showlegend=False),
                  row=1, col=col)
    # 끝점
    fig.add_trace(go.Scatter(x=[accum.real[-1]], y=[accum.imag[-1]],
                             mode='markers',
                             marker=dict(color='#f0f6fc', size=9),
                             hoverinfo='skip', showlegend=False),
                  row=1, col=col)
    # 분석값 target (X 마커)
    arg = t_fix
    if abs(abs(arg) - W) < 1e-9:
        analytic = 0.5
    elif abs(arg) < W:
        analytic = 1.0
    else:
        analytic = 0.0
    fig.add_trace(go.Scatter(x=[analytic], y=[0.0],
                             mode='markers',
                             marker=dict(color=_BLUE, size=12, symbol='x'),
                             hoverinfo='skip', showlegend=False),
                  row=1, col=col)
    fig.add_annotation(
        text=f'recovered={accum[-1].real:+.3f}<br>target={analytic:.1f}',
        xref=f'x{col} domain' if col>1 else 'x domain',
        yref=f'y{col} domain' if col>1 else 'y domain',
        x=0.5, y=-0.04, xanchor='center', yanchor='top',
        showarrow=False, font=dict(color=_FG, size=10),
    )

fig.update_layout(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    font=dict(color=_FG, size=11),
    margin=dict(l=10, r=10, t=60, b=70),
    height=420, showlegend=False,
)
ax_pairs_p4 = [('xaxis', 'y'), ('xaxis2', 'y2'), ('xaxis3', 'y3'), ('xaxis4', 'y4')]
for ax_name, scale_target in ax_pairs_p4:
    fig.layout[ax_name].update(range=[-lim_p4, lim_p4],
                                gridcolor=_GRID, zerolinecolor=_AXIS, color=_FG,
                                scaleanchor=scale_target, scaleratio=1)
for ay in ['yaxis', 'yaxis2', 'yaxis3', 'yaxis4']:
    fig.layout[ay].update(range=[-lim_p4, lim_p4],
                          gridcolor=_GRID, zerolinecolor=_AXIS, color=_FG)
fig.show()

# 정확도 표
print('\n  t_fix     rect_W(t_fix)   recovered            |error|')
print('  ──────────────────────────────────────────────────────')
for t_fix in [-3.0, -2.5, -2.0, -1.0, 0.0, 1.0, 1.5, 2.0, 2.5, 3.0]:
    arrows = X_o * np.exp(1j * om_o * t_fix) * domega_inv / (2 * np.pi)
    rec = complex(np.sum(arrows))
    if abs(abs(t_fix) - W) < 1e-9:
        a = 0.5
    elif abs(t_fix) < W:
        a = 1.0
    else:
        a = 0.0
    err = abs(rec.real - a)
    print(f'  {t_fix:+5.2f}     {a:.3f}           ({rec.real:+.4f},{rec.imag:+.4f})    {err:.2e}')


  t_fix     rect_W(t_fix)   recovered            |error|
  ──────────────────────────────────────────────────────
  -3.00     0.000           (-0.0000,+0.0000)    1.31e-05
  -2.50     0.000           (-0.0131,-0.0000)    1.31e-02
  -2.00     0.500           (+0.4982,+0.0000)    1.84e-03
  -1.00     1.000           (+1.0002,-0.0000)    2.03e-04
  +0.00     1.000           (+1.0098,+0.0000)    9.80e-03
  +1.00     1.000           (+1.0002,+0.0000)    2.03e-04
  +1.50     1.000           (+1.0157,-0.0000)    1.57e-02
  +2.00     0.500           (+0.4982,-0.0000)    1.84e-03
  +2.50     0.000           (-0.0131,+0.0000)    1.31e-02
  +3.00     0.000           (-0.0000,-0.0000)    1.31e-05


**관찰**:
- 외부 `t_fix=-2.5`: trail이 작은 영역을 맴돌다 끝점도 0 근처. 모든 ω 성분이 cancel.
- 경계 `t_fix=-2`: 끝점 ≈ 0.5. 사다리꼴 적분의 자연스러운 결과 — Gibbs phenomenon의 일종.
- 내부 `t_fix=0`: 끝점 ≈ 1. trail이 크게 휘돌면서 누적되는 모양.
- 내부 `t_fix=1.5`: 끝점도 ≈ 1. `t_fix=0`과는 trail의 *모양*이 다름 (linear phase 효과의 전조 — Phase 5에서 다룸).

수치 표에서 모든 허수부가 ≈ 0 (실수 신호의 켤레쌍 cancel — paired_by_abs 정렬). 펄스 외부 오차는 sinc tail의 truncation 영향.

## (2-2) Linear phase = time shift

푸리에 변환의 *time shift property*:
$$
x(t - \alpha) \;\longleftrightarrow\; X(j\omega)\cdot e^{-j\omega\alpha}
$$

수식 차원에서는 자명하지만, 역변환 누적 시각화로 보면 *왜 그런지*가 그림에서 직접 보인다.

역변환 적분 안에 X 대신 `X(jω)·e^(-jωα)`를 넣으면 `e^(-jωα)·e^(jωt) = e^(jω(t-α))`. 즉 `t_fix`를 **t* + α**로 잡으면 정확히 원본 X에서 `t_fix = t*`로 잡은 적분과 *같은 화살표 시퀀스*가 만들어진다.

데모: α = 1.5. rect_W(t)를 오른쪽으로 1.5만큼 평행이동한 신호 = rect_W(t-1.5)는 |t-1.5|<2 즉 `-0.5 < t < 3.5`에서 1.

### 5.1 — 3D 시각화로 Linear phase의 정체를 본다

먼저 `X(jω)·e^(-jωα)`가 *어떻게 생겼는지* 3D로 본다.

**(a) 원본 sinc `X(jω)`** — 짝함수라 순수 실수, ω축을 따라가는 *평면* 위 곡선 (Im X = 0, flatten_noise 작동).

In [15]:
ALPHA = 1.5  # 평행이동량
omegas_3d = np.linspace(-6*np.pi, 6*np.pi, 401)
X_3d_orig = 2 * np.sin(omegas_3d * W) / np.where(np.abs(omegas_3d) < 1e-9, 1.0, omegas_3d)
X_3d_orig = np.where(np.abs(omegas_3d) < 1e-9, 2*W, X_3d_orig).astype(complex)

panel_orig = SpectrumPlot(view='re_im', show_inset=False)
panel_orig.show_spectrum(omegas_3d, X_3d_orig)
panel_orig.figure.layout.title.text = '(5.1a) X(jω) = 2sin(ωW)/ω — 짝함수, Im X = 0 평면'
panel_orig.figure.show()

**(b) Linear phase `e^(-jωα)`** — 단위 헬릭스. ω 따라 `(cos(ωα), -sin(ωα))`로 일정한 각속도 -α로 회전. envelope = 1 (단위원통)

In [16]:
# 단위 헬릭스를 SpectrumPlot 좌표계 (ω, Re, Im)에 띄움
unit_phase = np.exp(-1j * omegas_3d * ALPHA)
panel_phase = SpectrumPlot(view='re_im', show_inset=False)
panel_phase.show_spectrum(omegas_3d, unit_phase)
panel_phase.figure.layout.title.text = (
    f'(5.1b) e^(-jω·{ALPHA}) — 단위 헬릭스, 각속도 -{ALPHA}'
)
panel_phase.figure.show()

**(c) `X(jω)·e^(-jωα)`** — 핵심 그림. carrier는 (b)의 헬릭스 모양 그대로지만 *반지름이 sinc로 변조*된다. envelope `|sinc|`를 ω축에 회전시킨 회전체로 함께 그려 헬릭스가 그 안에 정확히 들어차는 모습이 보임. envelope의 zero crossing(ω = nπ/W)에서 회전체가 ω축에 닿음.

In [17]:
# X(jω)·e^(-jωα): WoundSpectrumPlot 한 줄로
X_3d_shift = X_3d_orig * np.exp(-1j * omegas_3d * ALPHA)   # 5.2에서 재사용

panel_wound = WoundSpectrumPlot()
panel_wound.show_wound_spectrum(omegas_3d, X_3d_orig, alpha=ALPHA,
    label=f'(5.1c) X(jω)·e^(-jω·{ALPHA}) — sinc envelope 회전체 + carrier helix')
panel_wound.figure.show()

print(f'envelope max = {float(np.max(np.abs(X_3d_orig))):.3f}  (= 2W = {2*W})')
print(f'envelope zeros at ω = ±π/W, ±2π/W, ... = ±{np.pi/W:.3f}, ±{2*np.pi/W:.3f}, ...')

envelope max = 4.000  (= 2W = 4.0)
envelope zeros at ω = ±π/W, ±2π/W, ... = ±1.571, ±3.142, ...


**메시지**: helix(초록)가 envelope(주황 회전체) 안에 정확히 들어차 있다. envelope이 0에 닿는 곳(ω = ±π/W ≈ ±1.57)에서 helix도 ω축으로 모이고, envelope의 부호가 바뀌는 부분에서 helix가 반대편 회전 방향으로 넘어간다 (sinc가 음수가 되니 phase가 π 점프).

### (2-2) 후속 — Partial 적분을 3D로 (Trail A/B/C 비교, 세 케이스)

이제 역변환 적분을 ω 누적으로 본다. 좌표: x = ω, (y, z) = (Re partial, Im partial). ω가 -Ω에서 +Ω까지 진행하면서 누적 trail이 3D 공간에서 어떻게 자라는지 본다.

각 t에 대해 세 trail을 *한 그림*에 함께 그려 비교:

| Trail | 적분 핵 | t | 끝점 |
|---|---|---|---|
| A (초록) | `X(jω) · e^(jωt)` | t | rect_W(t) |
| B (주황) | `X(jω)·e^(-jωα) · e^(jωt)` | t | rect_W(t-α) |
| C (흰 점선) | `X(jω)·e^(-jωα) · e^(jω(t+α))` | t+α | rect_W(t)  → A와 정확히 같은 trail |

세 t 케이스: **rect_W(t) = 1, 0.5, 0** 각각 (펄스 안 / 경계 / 외부)
- t = 0: A = rect_W(0) = 1, B = rect_W(-1.5) = 1   → 끝점 같음, trail 모양 다름
- t = 2: A = rect_W(2) = 0.5 (경계), B = rect_W(0.5) = 1 (안)  → 끝점도 다름
- t = 3: A = rect_W(3) = 0 (외부), B = rect_W(1.5) = 1 (안)  → 끝점이 정반대

A vs C는 모든 t에서 정확히 같음 (time shift property).

In [18]:
def rect_W(t):
    if abs(abs(t) - W) < 1e-9:
        return 0.5
    return 1.0 if abs(t) < W else 0.0


def show_three_trails(t_value: float, label: str):
    """주어진 t에서 Trail A/B/C 세 개를 PartialIntegralComparisonPlot으로."""
    target_A = rect_W(t_value)
    target_B = rect_W(t_value - ALPHA)

    panel = PartialIntegralComparisonPlot()
    panel.show_trails(omegas_3d, [
        {'kernel': X_3d_orig,  't': t_value, 'width': 5,
         'color': _GREEN,
         'label': f'A: X(jω), t={t_value}  → {target_A}'},
        {'kernel': X_3d_shift, 't': t_value, 'width': 4,
         'color': _ORANGE,
         'label': f'B: X·e^(-jωα), t={t_value}  → {target_B}'},
        {'kernel': X_3d_shift, 't': t_value + ALPHA, 'width': 3, 'dash': 'dash',
         'color': '#f0f6fc', 'marker_symbol': 'cross',
         'label': f'C: X·e^(-jωα), t+α={t_value+ALPHA}  (A와 일치)'},
    ], label=label)
    panel.figure.show()

    # 끝점 검증 (재계산)
    domega = float(omegas_3d[1] - omegas_3d[0])
    cum_A = np.cumsum(X_3d_orig  * np.exp(1j * omegas_3d * t_value)         * domega / (2*np.pi))
    cum_B = np.cumsum(X_3d_shift * np.exp(1j * omegas_3d * t_value)         * domega / (2*np.pi))
    cum_C = np.cumsum(X_3d_shift * np.exp(1j * omegas_3d * (t_value+ALPHA)) * domega / (2*np.pi))
    diff_AC = float(np.max(np.abs(cum_A - cum_C)))
    print(f'[{label}]  t = {t_value}')
    print(f'  Trail A 끝점:    {cum_A[-1]:+.4f}    target rect_W(t)   = {target_A}')
    print(f'  Trail B 끝점:    {cum_B[-1]:+.4f}    target rect_W(t-α) = {target_B}')
    print(f'  Trail C 끝점:    {cum_C[-1]:+.4f}    target rect_W(t)   = {target_A}')
    print(f'  cum_A vs cum_C max diff = {diff_AC:.2e}')
    print()


# 케이스 1: t=0 — 펄스 안 (A=B=1, trail 모양만 다름)
show_three_trails(0.0, '(5.2a) t = 0 — 펄스 안 (A=B=1, trail 모양만 다름)')

[(5.2a) t = 0 — 펄스 안 (A=B=1, trail 모양만 다름)]  t = 0.0
  Trail A 끝점:    +0.9832+0.0000j    target rect_W(t)   = 1.0
  Trail B 끝점:    +1.0379-0.0000j    target rect_W(t-α) = 1.0
  Trail C 끝점:    +0.9832-0.0000j    target rect_W(t)   = 1.0
  cum_A vs cum_C max diff = 5.56e-17



**(5.2a) t=0**: A와 B 모두 펄스 안이라 *끝점은 같지만 trail 모양이 다르다*. linear phase가 곱해진 B는 ω 따라 더 회전된 경로로 누적된 뒤 같은 점에 도달. C는 t→t+α=1.5 보정으로 A와 정확히 일치 (흰 점선이 초록 trail 위에 덧칠).

In [19]:
# 케이스 2: t=2 — 펄스 경계 (A=0.5, B=1, 다른 끝점)
show_three_trails(2.0, '(5.2b) t = 2 — A는 펄스 경계 (=0.5), B는 펄스 안 (=1)')

[(5.2b) t = 2 — A는 펄스 경계 (=0.5), B는 펄스 안 (=1)]  t = 2.0
  Trail A 끝점:    +0.4958-0.0000j    target rect_W(t)   = 0.5
  Trail B 끝점:    +1.0179+0.0000j    target rect_W(t-α) = 1.0
  Trail C 끝점:    +0.4958-0.0000j    target rect_W(t)   = 0.5
  cum_A vs cum_C max diff = 2.48e-16



**(5.2b) t=2**: A는 펄스 경계라 끝점 ≈ 0.5, B는 t-α=0.5에서 펄스 안 → 끝점 ≈ 1. **두 trail이 다른 점에 도달**. C는 t+α=3.5에서 펄스 경계라 A와 같은 0.5에서 만남 — 시각적으로도 A 위에 정확히 덧칠된다.

In [20]:
# 케이스 3: t=3 — 펄스 외부 (A=0, B=1, 정반대)
show_three_trails(3.0, '(5.2c) t = 3 — A는 펄스 외부 (=0), B는 펄스 안 (=1)')

[(5.2c) t = 3 — A는 펄스 외부 (=0), B는 펄스 안 (=1)]  t = 3.0
  Trail A 끝점:    +0.0135-0.0000j    target rect_W(t)   = 0.0
  Trail B 끝점:    +1.0379+0.0000j    target rect_W(t-α) = 1.0
  Trail C 끝점:    +0.0135-0.0000j    target rect_W(t)   = 0.0
  cum_A vs cum_C max diff = 1.36e-16



**(5.2c) t=3**: 가장 강력한 메시지 — A는 펄스 *밖*이라 끝점 ≈ 0, B는 t-α=1.5라 펄스 안 → 끝점 ≈ 1. **두 trail이 정반대 위치에 도달**. C는 t+α=4.5에서 외부라 A와 같은 0에서 만남.

세 케이스를 종합하면 linear phase의 효과가 명확히 보인다 — *같은 t에서 보면 A와 B는 ω 따라 다른 회전을 누적하다 다른 끝점에 도달하지만, t를 α만큼 보정하면 두 적분이 항상 정확히 같다*. 끝점이 같든 다르든, A vs C의 동등성은 모든 t에서 보존된다 (time shift property).

**평행이동된 신호 전체 검증** — `X·e^(-jωα)`를 여러 t_fix에서 역변환해 평행이동된 rect를 복원:

In [21]:
# X_o_shift는 5.2 셀에서 X_3d_shift와 별개로 inverse 정확도용 데이터.
# Phase 4에서 만든 om_o, X_o (paired_by_abs 정렬)를 재사용.
X_o_shift = X_o * np.exp(-1j * om_o * ALPHA)

test_t_fix = [-1.5, -0.5, 0.0, 1.5, 3.0, 3.5, 4.5]

print('  t_fix     rect_W(t_fix - α)    recovered           |error|')
print('  ──────────────────────────────────────────────────────────')
for t_fix in test_t_fix:
    arrows = X_o_shift * np.exp(1j * om_o * t_fix) * domega_inv / (2 * np.pi)
    rec = complex(np.sum(arrows))
    arg = t_fix - ALPHA
    if abs(abs(arg) - W) < 1e-9:
        a = 0.5
    elif abs(arg) < W:
        a = 1.0
    else:
        a = 0.0
    err = abs(rec.real - a)
    print(f'  {t_fix:+5.2f}     {a:.3f}              ({rec.real:+.4f},{rec.imag:+.4f})    {err:.2e}')

  t_fix     rect_W(t_fix - α)    recovered           |error|
  ──────────────────────────────────────────────────────────
  -1.50     0.000              (-0.0000,+0.0000)    1.31e-05
  -0.50     0.500              (+0.4982,+0.0000)    1.84e-03
  +0.00     1.000              (+1.0157,+0.0000)    1.57e-02
  +1.50     1.000              (+1.0098,+0.0000)    9.80e-03
  +3.00     1.000              (+1.0157,-0.0000)    1.57e-02
  +3.50     0.500              (+0.4982,-0.0000)    1.84e-03
  +4.50     0.000              (-0.0000,-0.0000)    1.31e-05


**결과 해석**:
- 펄스 내부 (`t_fix - α ∈ (-W, W)` = `t_fix ∈ (-0.5, 3.5)`): recovered ≈ 1
- 경계 (`t_fix = -0.5` 또는 `3.5`): recovered ≈ 0.5
- 외부: recovered ≈ 0
- 모든 허수부 ≈ 0 (실수 신호 cancel)

평행이동된 rect가 정확히 복원됨. linear phase가 곧 time shift라는 property가 수치로도 검증.

## 정리

이 노트북에서 `rect_W(t)`(W=2)를 *순방향과 역방향* 양쪽으로 분석했다.

| 섹션 | 메시지 |
|---|---|
| (1)[1] | 적분의 3D 시각화 — 시간 도메인의 4-패널 |
| (1)[2] | 복소평면 단면 — ω 따라 호의 모양과 벡터 간격 변화 |
| (1)[3] | `e^{jθ}dθ`는 호 위 미소 변위를 1/j로 회전시킨 것. 미소 벡터 합 = 호 양 끝 변위 |
| (1)[4] | chord 길이 = `2sin(ωW)` — 단위원 + 직각 삼각형으로 직접 유도 |
| (1)[5] | ωW를 0→4π 슬라이더 — sin이 진동하는 모습 |
| (1)[6] | 1/ω 야코비안 다시 곱하면 `2sin(ωW)/ω = sinc` |
| (1)[7] | SpectrumPlot으로 전체 sinc 모양 검증 |
| (2-1) | sinc → rect 역변환. 펄스 내부/경계/외부에서 trail 모양 비교 |
| (2-2) | Linear phase × X(jω) = 평행이동된 신호. Trail A/B/C의 *같은 화살표 시퀀스*로 시각 증명 |

다른 노트북:
- [`01_intro.ipynb`](./01_intro.ipynb) — 복소 정현파/삼각함수/지수함수의 FT/IFT 기하적 직관 및 수치적 해석
- [`02_inverse.ipynb`](./02_inverse.ipynb) — FT, IFT 켤레 성질, 회전이동-평행이동 성질의 기하적 증명